# Retail Sales & Customer Insights Analytics

## tl;dr

The validated synthetic dataset contains **4,180 order lines** and **3,922 orders**. It reconciles to **₹160.1M revenue**, **₹41.3M profit**, a **25.8% margin** and an **8.8% order return rate**. Electronics leads revenue; Beauty & Personal Care leads margin and return rate; margin declines with deeper discounts.


## Context & Methods

This companion notebook audits the SQLite star schema, profiles data quality, reproduces KPI results and creates focused category and monthly views.

### Key Assumptions

- Data is deterministic and synthetic.
- Fact grain is one row per order line.
- Revenue is net of discount and return amount.
- Order metrics use first-line flags to prevent double counting.


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATABASE = PROJECT_ROOT / 'data' / 'database' / 'retail_analytics.db'
print(f'Project root: {PROJECT_ROOT}')


Project root: /workspace/scratch/46c9c99777d1/Retail_Sales_Customer_Insights_Analytics


## Data

### 1. Load the enriched star-schema view

In [2]:
with sqlite3.connect(DATABASE) as connection:
    sales = pd.read_sql_query('SELECT * FROM vw_sales_enriched', connection, parse_dates=['date'])
    quality_checks = pd.read_csv(PROJECT_ROOT / 'data' / 'quality' / 'data_quality_checks.csv')
print(f'Sales shape: {sales.shape}')
print(f'Date range: {sales.date.min().date()} to {sales.date.max().date()}')
print(f'Quality checks passed: {int(quality_checks.passed.sum())}/{len(quality_checks)}')


Sales shape: (4180, 27)
Date range: 2024-01-01 to 2025-12-31
Quality checks passed: 15/15


### 2. Reconcile headline KPIs

In [3]:
with sqlite3.connect(DATABASE) as connection:
    kpis = pd.read_sql_query(
        '''SELECT ROUND(SUM(revenue),2) AS revenue, ROUND(SUM(profit),2) AS profit,
                  ROUND(SUM(profit)/SUM(revenue),6) AS profit_margin,
                  SUM(order_flag) AS orders, SUM(return_order_flag) AS returned_orders,
                  ROUND(1.0*SUM(return_order_flag)/SUM(order_flag),6) AS return_rate
           FROM fact_sales''',
        connection,
    )
print(kpis.to_string(index=False))
assert float(kpis.loc[0, 'revenue']) == 160_100_000.0
assert float(kpis.loc[0, 'profit']) == 41_300_000.0
assert int(kpis.loc[0, 'orders']) == 3_922


    revenue     profit  profit_margin  orders  returned_orders  return_rate
160100000.0 41300000.0       0.257964    3922              345     0.087965


## Results

### 3. Monthly performance and target attainment

In [4]:
with sqlite3.connect(DATABASE) as connection:
    monthly = pd.read_sql_query(
        '''SELECT f.year_month, SUM(f.revenue) revenue, SUM(f.profit) profit,
                  SUM(f.order_flag) orders, t.target_revenue,
                  SUM(f.revenue)/t.target_revenue target_attainment
           FROM fact_sales f JOIN monthly_targets t USING(year_month)
           GROUP BY f.year_month, t.target_revenue ORDER BY f.year_month''',
        connection,
    )
print(monthly.head(6).round(3).to_string(index=False))
print(f'Months at or above target: {(monthly.target_attainment >= 1).sum()} of {len(monthly)}')


year_month    revenue     profit  orders  target_revenue  target_attainment
   2024-01 4996345.33 1182289.86     115      4796491.52              1.042
   2024-02 5418647.85 1420029.71     124      5310274.89              1.020
   2024-03 5556537.53 1421677.58     147      5556537.53              1.000
   2024-04 5706828.23 1523654.39     135      5820964.79              0.980
   2024-05 4624355.55 1280536.96     123      4809329.77              0.962
   2024-06 6304328.31 1654016.36     139      6682588.01              0.943
Months at or above target: 8 of 24


In [5]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(monthly.year_month, monthly.revenue / 1_000_000, color='#2563EB', marker='o', label='Revenue')
ax.plot(monthly.year_month, monthly.profit / 1_000_000, color='#D4A72C', marker='s', label='Profit')
ax.set_title('Monthly Revenue and Profit')
ax.set_ylabel('₹ million')
ax.tick_params(axis='x', rotation=55)
ax.legend(frameon=False)
fig.tight_layout()
chart_path = PROJECT_ROOT / 'reports' / 'charts' / 'notebook_monthly_revenue_profit.png'
fig.savefig(chart_path, dpi=160, bbox_inches='tight')
plt.close(fig)
print(f'Saved chart: {chart_path}')


Saved chart: /workspace/scratch/46c9c99777d1/Retail_Sales_Customer_Insights_Analytics/reports/charts/notebook_monthly_revenue_profit.png


### 4. Category performance

In [6]:
with sqlite3.connect(DATABASE) as connection:
    category = pd.read_sql_query(
        '''SELECT p.category, SUM(f.revenue) revenue, SUM(f.profit) profit,
                  SUM(f.profit)/SUM(f.revenue) profit_margin,
                  1.0*SUM(f.return_order_flag)/SUM(f.order_flag) return_rate
           FROM fact_sales f JOIN dim_product p USING(product_key)
           GROUP BY p.category ORDER BY revenue DESC''',
        connection,
    )
print(category.round(4).to_string(index=False))


              category     revenue      profit  profit_margin  return_rate
           Electronics 70577599.10 14843690.65         0.2103       0.0850
        Home & Kitchen 47384813.92 11561376.15         0.2440       0.0824
     Sports & Outdoors 19728748.90  6282731.34         0.3185       0.0883
               Fashion 14759353.18  5522757.75         0.3742       0.0914
Beauty & Personal Care  7649484.90  3089444.11         0.4039       0.0976


### 5. RFM segments and discount profitability

In [7]:
with sqlite3.connect(DATABASE) as connection:
    rfm = pd.read_sql_query(
        'SELECT rfm_segment, COUNT(*) customers, SUM(monetary) revenue FROM customer_rfm GROUP BY rfm_segment ORDER BY customers DESC',
        connection,
    )
    discount = pd.read_sql_query(
        '''SELECT CASE WHEN discount_pct=0 THEN 'No Discount' WHEN discount_pct<=.10 THEN '1-10%'
                     WHEN discount_pct<=.20 THEN '11-20%' ELSE 'Above 20%' END discount_band,
                  SUM(revenue) revenue, SUM(profit)/NULLIF(SUM(revenue),0) profit_margin
           FROM fact_sales GROUP BY discount_band ORDER BY profit_margin DESC''',
        connection,
    )
print('RFM segments:')
print(rfm.round(2).to_string(index=False))
print()
print('Discount profitability:')
print(discount.round(4).to_string(index=False))


RFM segments:
        rfm_segment  customers     revenue
        Hibernating        268 18361350.25
            At Risk        194 29548420.88
    Loyal Customers        186 31290639.62
          Champions        176 45705936.77
Potential Loyalists        160 19446128.18
    Needs Attention        149 14589286.59
      New Customers         21  1158237.71

Discount profitability:
discount_band     revenue  profit_margin
  No Discount 32336448.31         0.3450
        1-10% 68482576.63         0.2837
       11-20% 45907131.51         0.2040
    Above 20% 13373843.55         0.1012


## Takeaways

- Electronics is the scale category; higher-margin categories contribute less revenue.
- Beauty & Personal Care combines the highest category margin with the highest return rate.
- Champions and At-Risk customers both represent meaningful revenue pools.
- Deep discount bands are associated with lower descriptive profit margins.
- The star schema, KPI arithmetic and quality outputs reconcile across independent query paths.

**Caveat:** these patterns are created from synthetic data and demonstrate methods rather than real-world causal evidence.
